
## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [7]:

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x116be82f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x116be8d70>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [18]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details."""
    title:str = Field(...,description="Title of the movie")
    year:int = Field(..., description="Release Date of the movie")
    director:str= Field(..., description="The director of the movie")
    rating:float = Field(..., director="The Rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("I want to details of Iron Man movie")
print(response["parsed"])


/var/folders/3s/4_rcnc8s0rng9wsklkfgs4mw0000gn/T/ipykernel_953/2566064288.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'director'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  rating:float = Field(..., director="The Rating of the movie")


title='Iron Man' year=2008 director='Jon Favreau' rating=7.9


In [29]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    releaseCountries:list[str]
    budget : float | None = Field(None, description="Budget of the movie")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("please provide the details about Iron Man movie")
print(response)



title='Iron Man' year=2008 cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Gwyneth Paltrow', role='Pepper Potts'), Actor(name='Terrence Howard', role='James Rhodes / War Machine')] genres=['Action', 'Sci-Fi', 'Adventure'] releaseCountries=['United States'] budget=None
